# A3 · Exploring the messy export (EDA)

**You are the detective.** `data/raw_emr_export.csv` is what the hospital handed over. **Eight kinds of data problem** are hidden in it. Find as many as you can.

**Rules**
1. **Look, don't change.** Nothing in this notebook modifies the data; fixing happens in A4. EDA that "fixes as it goes" destroys the evidence.
2. **Write every finding in the problem log** (next cell) with *numbers*: how many rows, what proportion, an example.
3. **Don't open** `slotpred/_messy_export.py` (spoiler) or `data/_hidden_truth.csv` (that's for the oracle only, never for a model).

Each check has a short **why**, given code where it's plumbing, and `TODO(you)` where the looking is yours.

## Problem log (fill this in as you go)

| # | Problem | Evidence (numbers, example) | Rows affected | Likely cause | Fix to try in A4 |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |
| 5 | | | | | |
| 6 | | | | | |
| 7 | | | | | |
| 8 | | | | | |

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from slotpred.eda import load_raw, log_hist, monthly_median

raw = load_raw()
print(raw.shape)

## Check 1 - Shape and types

**Why:** a column with the wrong type hides problems. Dates stored as text can't be compared; a number stored as float may be hiding missing values (NaN forces int to float).

In [ ]:
display(raw.head())
raw.dtypes

**Questions:** Which columns are text but should be dates or times? Why might `age` be a float when ages are whole numbers?

*Answer:*

## Check 2 - Keys and duplicates

**Why:** `appointment_id` should identify exactly one visit. Duplicates count a visit twice: double weight in training, and if one copy lands in train and the other in test, the model is tested on a visit it has already seen (a form of leakage).

In [ ]:
# TODO(you):
# 1. How many rows are EXACT duplicates of another row?           hint: raw.duplicated()
# 2. How many appointment_ids appear more than once?              hint: raw["appointment_id"].duplicated()
# 3. Are the repeated ids always exact copies, or do some differ?  compare 1 and 2

## Check 3 - Missing values

**Why:** *how many* is missing matters, but *which rows* matters more:

| Type | Meaning | Example | Danger |
|---|---|---|---|
| **MCAR** (missing completely at random) | Missingness unrelated to anything | A random export glitch | Low: dropping rows is fine |
| **MAR** (missing at random, given other columns) | Depends on something you can see | One clinic never records age | Medium: fill using that column |
| **MNAR** (missing not at random) | Depends on the missing value itself | End time missing because the visit was so long the doctor forgot | High: dropping biases the data |

In [ ]:
# TODO(you):
# 1. Missing count AND percentage per column.                      hint: raw.isna().sum(), raw.isna().mean()
# 2. For each column with missing values: is the missing rate the same in every clinic? every month?
#    hint: raw.assign(missing=raw["COL"].isna()).groupby("clinic")["missing"].mean()

## Check 4 - Categories

**Why:** "Dr. Deepak Rao", "DR DEEPAK RAO" and "Deepak R." are one doctor to a human and three doctors to a model, each with a third of the data.

**Reference from the hospital:** 21 doctors, 8 departments, 5 clinics, 3 visit types (`new`, `follow-up`, `review`). Every doctor belongs to exactly one department.

In [ ]:
# TODO(you):
# 1. Number of distinct values in doctor, department, clinic, visit_type. Compare with the reference.   hint: .nunique()
# 2. Look at the actual values of any column that has too many.                                          hint: .value_counts()
# 3. Does every doctor appear in exactly one department?    hint: raw.groupby("doctor")["department"].nunique()

## Check 5 - Numbers: impossible and placeholder values

**Why:** systems often store "unknown" as a fake number (999, -1, 0). A model treats 999 as a real, very old patient.

In [ ]:
# TODO(you):
# 1. raw.describe() for the numeric columns. Anything impossible?
# 2. Look at the most common values and the extremes of age.      hint: .value_counts().head(), .sort_values()
# 3. Is prior_reports sensible for every visit type?               hint: groupby visit_type, describe

## Check 6 - The target: consult minutes

**Why:** the EMR has no "duration" column. You derive the target from two timestamps, and every problem in those timestamps becomes a problem in the target.

### Outliers: two ways to define "suspicious"

**Statistical (IQR rule):** $\text{IQR} = Q_3 - Q_1$; flag values below $Q_1 - 1.5\,\text{IQR}$ or above $Q_3 + 1.5\,\text{IQR}$.
Consult times are right-skewed, so on the raw scale the rule flags many *genuine* long visits. Apply it to **log(minutes)** instead, where the skew is gone.

**Domain rule:** "a consult cannot take under 1 minute or over 2 hours". Statistics tell you *what's unusual*; only domain knowledge tells you *what's impossible*. Unusual is not the same as wrong.

In [ ]:
# TODO(you): build the target, without changing `raw`
# 1. Parse consult_start and consult_end with pd.to_datetime.
# 2. minutes = (end - start) in minutes.          hint: .dt.total_seconds() / 60
# 3. How many are NaN? negative? under 1 minute? over 120 minutes?

minutes = None   # replace

In [ ]:
# Given: the shape of the target on a log scale (run after your cell above)
log_hist(minutes, title="consult minutes, all rows")
plt.show()

In [ ]:
# TODO(you):
# 1. IQR fences on minutes, then on np.log(minutes). How many rows does each flag? Which do you trust more, and why?
# 2. Print the 10 shortest and 10 longest consults with all their columns. What do they have in common?

## Check 7 - Consistency over time, places and timestamps

**Why:** a process change (new software, new reception staff, a new rule) shows up as a **step** in a time plot. A model trained across the step learns a mixture of two different processes.

In [ ]:
# Given: monthly median consult minutes for each clinic (needs `minutes` from Check 6)
table = monthly_median(raw.assign(minutes=minutes), "minutes", by="clinic")
plt.show()
table.round(1)

In [ ]:
# TODO(you): the timestamps should follow   check_in  <=  consult_start  <  consult_end
# 1. How many rows break each part of that order? How many have consult_start EXACTLY equal to check_in?
# 2. Where and when do those rows happen?     hint: groupby clinic and month

## Check 8 - Leakage smells

**Why:** a feature is only allowed if it is **known when the slot is booked**, days before the visit. Fill in the table: for each column, is it known at booking? Is it the target or used to build it? Is it useless or risky?

| Column | Known at booking? | Use as a feature? | Why |
|---|---|---|---|
| appointment_id | | | |
| date, slot_time | | | |
| clinic, doctor, department | | | |
| patient_id, patient_name | | | |
| age, interpreter, prior_reports | | | |
| visit_type, teleconsult | | | |
| check_in | | | |
| consult_start, consult_end | | | |

Hint for `appointment_id`: are the ids random, or do they encode something else?

## Wrap-up

1. How many of the 8 problem kinds did you log?
2. For each one: is it **MCAR, MAR or MNAR**-like? Would simply dropping the rows bias the data?
3. Which problem would hurt the model most if you missed it?